# GLS Multi-Seed Training — Distributed Across Multiple Colab Accounts

Run this **same notebook on all 5 Google accounts simultaneously**.
Each account picks seeds that aren't already claimed or done — no conflicts.

### How it works
All accounts mount the **same Google Drive folder**. Before training a seed, each account:
1. Checks if a **checkpoint** exists → skip (already done by someone)
2. Checks if a **lock file** exists → skip (another account is working on it)
3. Otherwise → write a lock file → train → lock stays as a permanent marker

> **Before running:** Runtime → Change runtime type → **T4 GPU**

---
## ⚙️ Config — Set This Per Account

Each account needs a unique `ACCOUNT_ID` (0–4). That's the **only thing you change**.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  SET THIS TO A UNIQUE NUMBER PER ACCOUNT  (0, 1, 2, 3, 4)  ║
# ╚══════════════════════════════════════════════════════════════╝
ACCOUNT_ID = 0       # <-- change to 1, 2, 3, 4 on each account

# ── Shared settings (same on all accounts) ──────────────────────
NUM_SEEDS    = 10
NUM_ACCOUNTS = 5
EXPERIMENTS  = [
    "exp01_unet_noaug",
    "exp02_unet_aug",
    "exp03_attnunet_noaug",
    "exp04_attnunet_aug",
]

# ── Seed assignment (round-robin, e.g. account 0 → seeds 0, 5) ──
# This is the DEFAULT assignment. The lock-file system also lets
# one account pick up seeds from a crashed account automatically.
MY_SEEDS = list(range(ACCOUNT_ID, NUM_SEEDS, NUM_ACCOUNTS))
print(f"Account {ACCOUNT_ID} — assigned seeds: {MY_SEEDS}")

---
## 1. Setup

In [ ]:
import os

REPO_DIR = "/content/gls-lesion-segmentation"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/p4ntomath/gls-lesion-segmentation.git
else:
    print("Repo already cloned, pulling latest...")

%cd {REPO_DIR}
!git pull

In [ ]:
!pip install -r requirements.txt -q

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

### Mount Google Drive — All accounts must use the SAME Drive folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# All accounts share this single Drive folder
DRIVE_ROOT = "/content/drive/MyDrive/gls-data"

!mkdir -p {DRIVE_ROOT}/processed
!mkdir -p {DRIVE_ROOT}/splits
!mkdir -p {DRIVE_ROOT}/outputs/checkpoints
!mkdir -p {DRIVE_ROOT}/outputs/logs
!mkdir -p {DRIVE_ROOT}/outputs/results
!mkdir -p {DRIVE_ROOT}/outputs/figures
!mkdir -p {DRIVE_ROOT}/outputs/locks      # ← coordination folder

!rm -rf data/processed data/splits outputs
!ln -s {DRIVE_ROOT}/processed data/processed
!ln -s {DRIVE_ROOT}/splits    data/splits
!ln -s {DRIVE_ROOT}/outputs   outputs

print("✅ Drive mounted and symlinked")

---
## 2. Coordination Helpers

In [ ]:
import json
import subprocess
import sys
import time
import socket
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

CHECKPOINTS_DIR = Path("outputs/checkpoints")
RESULTS_DIR     = Path("outputs/results")
LOGS_DIR        = Path("outputs/logs")
LOCKS_DIR       = Path("outputs/locks")

# A crashed lock is considered stale after this many seconds
# (set to 0 to never auto-recover stale locks)
LOCK_STALE_SECONDS = 60 * 60 * 6   # 6 hours

HOSTNAME = socket.gethostname()     # unique per Colab VM


def _lock_path(experiment: str, seed: int) -> Path:
    return LOCKS_DIR / f"{experiment}_seed{seed:02d}.lock"

def checkpoint_exists(experiment: str, seed: int) -> bool:
    return (CHECKPOINTS_DIR / f"{experiment}_seed{seed:02d}.pt").exists()

def results_exist(experiment: str, seed: int) -> bool:
    return (RESULTS_DIR / f"{experiment}_seed{seed:02d}" / "results.json").exists()


def try_claim_seed(experiment: str, seed: int) -> bool:
    """
    Attempt to claim a seed for training.
    Returns True if this account successfully claimed it (or already owns it).
    Returns False if another account has it claimed or it's already done.
    """
    # Already done by someone — no need to claim
    if checkpoint_exists(experiment, seed):
        return False

    lock = _lock_path(experiment, seed)

    if lock.exists():
        # Read the lock to see who owns it and when
        try:
            info = json.loads(lock.read_text())
        except Exception:
            info = {}

        owner    = info.get("account_id")
        hostname = info.get("hostname", "")
        claimed  = info.get("claimed_at", 0)
        age      = time.time() - claimed

        # We own this lock (Colab restarted mid-run)
        if hostname == HOSTNAME:
            print(f"  🔄 Reclaiming own lock (same VM, age {age/3600:.1f}h)")
            return True

        # Stale lock from a VM that vanished — steal it
        if LOCK_STALE_SECONDS > 0 and age > LOCK_STALE_SECONDS:
            print(f"  ⚠️  Stale lock (account {owner}, {age/3600:.1f}h old) — stealing")
            # Fall through to write our own lock below
        else:
            print(f"  ⏭️  Seed locked by account {owner} ({age/60:.0f} min ago) — skipping")
            return False

    # Claim the seed
    LOCKS_DIR.mkdir(parents=True, exist_ok=True)
    lock.write_text(json.dumps({
        "account_id": ACCOUNT_ID,
        "hostname":   HOSTNAME,
        "claimed_at": time.time(),
        "claimed_at_str": datetime.now().isoformat(),
        "experiment": experiment,
        "seed":       seed,
    }, indent=2))
    return True


def release_lock(experiment: str, seed: int) -> None:
    """Remove the lock file after training (checkpoint is now the real marker)."""
    lock = _lock_path(experiment, seed)
    if lock.exists():
        lock.unlink()


def show_status(experiments=EXPERIMENTS, num_seeds=NUM_SEEDS):
    """Print a table showing done/locked/pending seeds for every experiment."""
    print(f"\n{'Experiment':<30} | {'Checkpoints':<25} | {'Results':<25} | {'Locked':<25}")
    print("-" * 115)
    for exp in experiments:
        ckpts   = [s for s in range(num_seeds) if checkpoint_exists(exp, s)]
        results = [s for s in range(num_seeds) if results_exist(exp, s)]
        locks   = []
        for s in range(num_seeds):
            lp = _lock_path(exp, s)
            if lp.exists() and not checkpoint_exists(exp, s):
                try:
                    info = json.loads(lp.read_text())
                    locks.append(f"s{s}→a{info.get('account_id','?')}")
                except Exception:
                    locks.append(f"s{s}→?")
        print(f"{exp:<30} | {str(ckpts):<25} | {str(results):<25} | {str(locks):<25}")


print("✅ Coordination helpers loaded")

In [ ]:
# ── Training / Evaluation helpers ───────────────────────────────────────────

def run_live(cmd: list) -> int:
    """Run subprocess and stream output live to the notebook."""
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    process.wait()
    return process.returncode


def train_seed(experiment: str, seed: int) -> bool:
    """
    Train one seed. Uses --resume so a partial run can be continued.
    Returns True on success.
    """
    tag = f"seed{seed:02d}"
    print(f"\n{'='*65}")
    print(f"  TRAIN  {experiment}  seed={seed}  account={ACCOUNT_ID}")
    print(f"{'='*65}")

    # If checkpoint already exists, just resume evaluation — no retraining
    if checkpoint_exists(experiment, seed):
        print("  ⏭️  Checkpoint already exists — skipping training")
        return True

    cmd = [
        sys.executable, "scripts/train.py",
        "--experiment", experiment,
        "--seed", str(seed),
        "--output-tag", tag,
        "--resume",
        "--allow-resume-config-mismatch",
    ]
    rc = run_live(cmd)
    if rc != 0:
        print(f"  ❌ Training failed (exit {rc})")
    return rc == 0


def evaluate_seed(experiment: str, seed: int, force: bool = False) -> bool:
    """Evaluate one seed. Skips if results already exist (unless force=True)."""
    tag = f"seed{seed:02d}"
    print(f"\n  EVAL   {experiment}  seed={seed}")

    if not force and results_exist(experiment, seed):
        print("  ⏭️  Results already exist — skipping evaluation")
        return True

    cmd = [
        sys.executable, "scripts/evaluate.py",
        "--experiment", experiment,
        "--output-tag", tag,
    ]
    rc = run_live(cmd)
    if rc != 0:
        print(f"  ❌ Evaluation failed (exit {rc})")
    return rc == 0


def run_my_seeds(experiment: str, seeds: list = None, force: bool = False):
    """
    Train + evaluate the seeds assigned to this account.
    Uses lock files so other accounts don't do the same seeds.
    """
    if seeds is None:
        seeds = MY_SEEDS

    trained_ok, eval_ok, skipped, failed = [], [], [], []

    for seed in seeds:
        tag = f"seed{seed:02d}"

        # ── Try to claim the seed ──────────────────────────────────────────
        claimed = try_claim_seed(experiment, seed)
        if not claimed:
            if checkpoint_exists(experiment, seed):
                print(f"\n  ✅ {experiment}_seed{seed:02d}: already done by someone")
            skipped.append(seed)
            continue

        # ── Train ─────────────────────────────────────────────────────────
        ok = train_seed(experiment, seed)

        if ok:
            trained_ok.append(seed)
            # Checkpoint is now on Drive — release lock (checkpoint is the
            # real permanent marker; lock file was just a "working on it" signal)
            release_lock(experiment, seed)
        else:
            failed.append(seed)
            # Keep lock so other accounts don't retry a broken seed
            # (user can manually delete the lock to allow retry)
            continue

        # ── Evaluate ──────────────────────────────────────────────────────
        ok_e = evaluate_seed(experiment, seed, force=force)
        if ok_e:
            eval_ok.append(seed)
        else:
            failed.append(f"eval-{seed}")

    print(f"\n{'─'*50}")
    print(f"  {experiment}  account={ACCOUNT_ID}  summary:")
    print(f"    Trained+Evaluated : {trained_ok}")
    print(f"    Skipped (done/locked): {skipped}")
    if failed:
        print(f"    ❌ Failed         : {failed}")
    return trained_ok


print("✅ Training helpers loaded")

---
## 3. Check Current Status (run anytime)
Shows which seeds have checkpoints, results, or active locks across all accounts.

In [ ]:
show_status()

---
## 4. Run My Seeds

Each account runs only its assigned seeds. Lock files prevent double-work.

**Default seed assignment (round-robin):**
| Account | Seeds |
|---------|-------|
| 0 | 0, 5 |
| 1 | 1, 6 |
| 2 | 2, 7 |
| 3 | 3, 8 |
| 4 | 4, 9 |

If an account finishes early and others are still running, uncomment the **Fill-in** cell to pick up unclaimed seeds.

### Experiment 1 — U-Net, No Augmentation

In [ ]:
run_my_seeds("exp01_unet_noaug")

### Experiment 2 — U-Net, With Augmentation

In [ ]:
run_my_seeds("exp02_unet_aug")

### Experiment 3 — Attention U-Net, No Augmentation

In [ ]:
run_my_seeds("exp03_attnunet_noaug")

### Experiment 4 — Attention U-Net, With Augmentation

In [ ]:
run_my_seeds("exp04_attnunet_aug")

### (Optional) Fill-in: Pick up any unclaimed seeds
Run this if you finished early and want to help other accounts.

In [ ]:
# Uncomment to pick up any unclaimed seeds for an experiment:
# run_my_seeds("exp01_unet_noaug", seeds=list(range(NUM_SEEDS)))

---
## 5. Aggregate + Compare
Run this on **any single account** after all 10 seeds are done.

In [ ]:
def aggregate_results(experiment: str, num_seeds: int = NUM_SEEDS) -> dict | None:
    all_summaries, all_per_image = [], []
    for seed in range(num_seeds):
        path = RESULTS_DIR / f"{experiment}_seed{seed:02d}" / "results.json"
        if not path.exists():
            print(f"  ⚠️  Seed {seed:02d} missing")
            continue
        with open(path) as f:
            data = json.load(f)
        all_summaries.append(data.get("summary", {}))
        for row in data.get("per_image", []):
            row["seed"] = seed
            all_per_image.append(row)

    if not all_summaries:
        print(f"No results for {experiment}")
        return None

    agg = {"experiment": experiment, "num_seeds": len(all_summaries)}
    metrics = sorted({k for s in all_summaries for k, v in s.items() if isinstance(v, (int, float))})
    for m in metrics:
        vals = [s[m] for s in all_summaries if m in s]
        if vals:
            agg[f"{m}_mean"] = float(np.mean(vals))
            agg[f"{m}_std"]  = float(np.std(vals))
            agg[f"{m}_min"]  = float(np.min(vals))
            agg[f"{m}_max"]  = float(np.max(vals))

    out = RESULTS_DIR / f"{experiment}_aggregated"
    out.mkdir(parents=True, exist_ok=True)
    with open(out / "results_aggregated.json", "w") as f:
        json.dump(agg, f, indent=2)
    pd.DataFrame([agg]).to_csv(out / "results_aggregated.csv", index=False)
    print(f"✅ {experiment}: {len(all_summaries)} seeds aggregated → {out}/")
    return agg


# Aggregate all experiments
all_agg = {exp: aggregate_results(exp) for exp in EXPERIMENTS}

In [ ]:
# Summary table
rows = []
for exp in EXPERIMENTS:
    path = RESULTS_DIR / f"{exp}_aggregated" / "results_aggregated.json"
    if not path.exists():
        continue
    with open(path) as f:
        r = json.load(f)
    row = {"experiment": exp, "n_seeds": r.get("num_seeds")}
    for k, v in r.items():
        if k.endswith("_mean") and not k.startswith("per_image"):
            m = k[:-5]
            row[f"{m} (mean)"] = round(v, 4)
            row[f"{m} (std)"]  = round(r.get(f"{m}_std", 0), 4)
    rows.append(row)

if rows:
    df = pd.DataFrame(rows)
    print("\n📊 FINAL SUMMARY — All experiments (10 seeds each)")
    display(df)
    df.to_csv(RESULTS_DIR / "all_experiments_summary.csv", index=False)
    print("\n✅ Saved to outputs/results/all_experiments_summary.csv")
else:
    print("No aggregated results yet")

---
## 6. Wilcoxon Tests

In [ ]:
from scipy.stats import wilcoxon

def per_sample_scores(experiment, metric="dice", num_seeds=NUM_SEEDS):
    scores = {}
    for seed in range(num_seeds):
        path = RESULTS_DIR / f"{experiment}_seed{seed:02d}" / "results.json"
        if not path.exists():
            continue
        with open(path) as f:
            data = json.load(f)
        for row in data.get("per_image", []):
            sid = row.get("sample_id") or row.get("id")
            if sid and metric in row:
                scores.setdefault(sid, []).append(row[metric])
    return {sid: float(np.mean(v)) for sid, v in scores.items() if v}

comparisons = [
    ("exp01_unet_noaug",     "exp02_unet_aug",     "Augmentation effect on U-Net?"),
    ("exp03_attnunet_noaug", "exp04_attnunet_aug", "Augmentation effect on Attn U-Net?"),
    ("exp02_unet_aug",       "exp04_attnunet_aug", "U-Net vs Attn U-Net (augmented)?"),
]

print("=" * 70)
print("WILCOXON SIGNED-RANK TESTS  (metric: Dice)")
print("=" * 70)

for exp1, exp2, question in comparisons:
    d1, d2 = per_sample_scores(exp1), per_sample_scores(exp2)
    common = sorted(set(d1) & set(d2))
    if not common:
        print(f"\n⚠️  {question} — no common samples yet")
        continue
    s1 = np.array([d1[s] for s in common])
    s2 = np.array([d2[s] for s in common])
    if np.allclose(s1, s2):
        print(f"\n⚠️  {question} — scores identical")
        continue
    stat, pval = wilcoxon(s1, s2)
    print(f"\n❓ {question}")
    print(f"   {exp1:<35}: {s1.mean():.4f} ± {s1.std():.4f}")
    print(f"   {exp2:<35}: {s2.mean():.4f} ± {s2.std():.4f}")
    print(f"   n={len(common)},  W={stat:.2f},  p={pval:.4f}")
    if pval < 0.05:
        winner = exp2 if s2.mean() > s1.mean() else exp1
        print(f"   ✅ Significant (p<0.05) → {winner} is better")
    else:
        print(f"   ⚠️  Not significant (p≥0.05)")